# Chapter 11 - Frequency-Severity Techniques

On this page, we will recreating Chapter 11 of Friendland. There is not a dedicated ``FrequencySeverity`` method in the package. Instead, we will chain together existing methods and use ``Triangle`` arithmetic to calculate the utimate. 

We will begin by import packages and setting up some helper functions.

In [0]:
assert 1 == 2 
import numpy as np
import pandas as pd
import chainladder as cl
from IPython.display import display as nb_display

#create reported/paid summary exhibit
def summary_exh(reported_tri:cl.Triangle,paid_tri:cl.Triangle,ult:cl.Triangle) -> pd.DataFrame:
    output = reported_tri.latest_diagonal.to_frame()
    output.columns = ['Reported']
    output['Paid'] = paid_tri.latest_diagonal.to_frame()
    #using floating point offset to achieve standard rounding
    output['Ult Claims'] = (ult.latest_diagonal + 1e-9).round(0).to_frame()
    output['Case Outstanding'] = output["Reported"] - output["Paid"]
    output['IBNR'] = output["Ult Claims"] - output["Reported"]
    output['Unpaid'] = output["Ult Claims"] - output["Paid"]
    return output.round(0)

#create a dict of developed triangles for each of the selection assumptions on a given page
def average_dev(tri: cl.Triangle, avg_params: dict[str,int]) -> dict[cl.Triangle]:
    return {k:cl.Development(**v).fit_transform(tri) for k,v in avg_params.items()}

#combine a dict of triangles into a singla triangle
def combine_tri(devs: dict[cl.Triangle]) -> cl.Triangle:
    avgs = [v.rename('index',k) for k,v in devs.items()]
    return cl.concat(avgs,axis=0)

#combine the ldf_ of a dict of triangles into a singla triangle
def combine_ldf(devs: dict[cl.Triangle]) -> cl.Triangle:
    return combine_tri({k:v.ldf_ for k,v in devs.items()})

#create a dict of DisposalRate estimators for each of the selection assumptions on a given page
def average_dr(tri: cl.Triangle, w:cl.Triangle, avg_params: dict[str,int]) -> dict[cl.DisposalRate]:
    return {k:cl.DisposalRate(**v).fit(X=tri,sample_weight = w) for k,v in avg_params.items()}

#combine the disposal_rate_ of a dict of triangles into a singla triangle
def combine_disposal(devs: dict[cl.Triangle]) -> cl.Triangle:
    return combine_tri({k:v.disposal_rate_ for k,v in devs.items()})

#create a dict of weighted regression objects for each of the selection assumptions on a given page
def regs(tri: cl.Triangle, avg_params: dict[str,int]) -> dict[cl.WeightedRegression]:
    return {k:cl.WeightedRegression(
        axis=2,
        thru_orig=False,
        xp=tri.get_array_module()
    ).fit(
        np.ones(tri.shape) * tri.origin.year.values[None,None,:,None],
        np.log(tri.values),
        cl.TriangleWeight(**v).fit(tri).w_.values
    ) for k,v in avg_params.items()}

#combine all the trends and r-squared values into pandas DataFrame
def reg_outputs(regs: dict[cl.WeightedRegression],devs:pd.Series) -> pd.DataFrame:
    trend = {k:(np.exp(v.slope_)-1).flatten() for k,v in regs.items()}
    trend = pd.DataFrame.from_dict(trend,orient='index')
    trend.columns = devs
    rsq = {k:v.rsq_.flatten() for k,v in regs.items()}
    rsq = pd.DataFrame.from_dict(rsq,orient='index')
    rsq.columns = devs
    return trend,rsq

#create a dict of weighted regression objects for each of the selection assumptions on a given page
def average_sev(tri: cl.Triangle, avg_params: dict[str,int]) -> dict[cl.Triangle]:
    output = {}
    for k,v in avg_params.items():
        w = cl.TriangleWeight(**v).fit(tri)
        weighted_tri = tri * w.w_.values
        output[k] = weighted_tri.mean()
    return output

## Exhibit I Analysis

Displaying all the triangles and dataframes take up quite a bit of code. We will distill the core analysis code at the top of each exhibit. 

Exhibit I follows Approach 1. Count and Severity are developed separately and multiplied together. 

In [0]:
#loading data and assumptions
e1_tri = cl.load_sample('friedland_auto_freq_sev')
e1_cnt_assumptions = {}
e1_cnt_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e1_cnt_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e1_cnt_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e1_cnt_assumptions['volume_5'] = {'n_periods':5, 'average':'volume'}
e1_cnt_assumptions['volume_3'] = {'n_periods':3, 'average':'volume'}
e1_sev_assumptions = {}
e1_sev_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e1_sev_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e1_sev_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}

#developing closed claim counts
e1_ccc_devs = average_dev(e1_tri['Closed Claim Counts'],e1_cnt_assumptions)
e1_ccc_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e1_ccc_devs['simple_3'])
e1_ccc_selected.ldf_ = e1_ccc_selected.ldf_.round(3)

#developing reported claim counts
e1_rcc_devs = average_dev(e1_tri['Reported Claim Counts'],e1_cnt_assumptions)
e1_rcc_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e1_rcc_devs['simple_3'])
e1_rcc_selected.ldf_ = e1_rcc_selected.ldf_.round(3)

#combining closed and reported claim counts
e1_ccc_cl = cl.Chainladder().fit(e1_ccc_selected)
e1_rcc_cl = cl.Chainladder().fit(e1_rcc_selected)
e1_cc_ult = (e1_ccc_cl.ultimate_ + e1_rcc_cl.ultimate_)/2
e1_cc_ult.iloc[:,:,-1,:] = e1_rcc_cl.ultimate_.iloc[:,:,-1,:]

#calculating and developing reported severity
e1_rsev = e1_tri['Reported Claims'] / e1_tri['Reported Claim Counts']
e1_rsev_devs = average_dev(e1_rsev,e1_sev_assumptions)
e1_rsev_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e1_rsev_devs['medial_5x1'])
e1_rsev_selected.ldf_ = e1_rsev_selected.ldf_.round(3)
e1_rsev_cl = cl.Chainladder().fit(e1_rsev_selected)

#combining developed count and severity
e1_ult = e1_rsev_cl.ultimate_ * e1_cc_ult / 1000

## Exhibit I Sheet 1

In [0]:
print('PART 1 - Data Triangle')
nb_display(e1_tri['Closed Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e1_tri['Closed Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e1_ccc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e1_ccc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e1_ccc_selected.cdf_.round(3))
print('Percent Closed')
nb_display(1/e1_ccc_selected.cdf_.round(3))

## Exhibit I Sheet 2

In [0]:
print('PART 1 - Data Triangle')
nb_display(e1_tri['Reported Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e1_tri['Reported Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e1_rcc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e1_rcc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e1_rcc_selected.cdf_.round(3))
print('Percent Reported')
nb_display(1/e1_rcc_selected.cdf_.round(3))

## Exhibit I Sheet 3

In [0]:
e1_ccc_df = cl.model_diagnostics(e1_ccc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e1_rcc_df = cl.model_diagnostics(e1_rcc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e1_s3 = e1_ccc_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Closed Claim Counts','CDF':'Closed CDF','Ultimate':'Ult Count Using CCC'})
e1_s3[['Reported Claim Counts','Reported CDF','Ult Count Using RCC']] = e1_rcc_df[['Latest','CDF','Ultimate']]
e1_s3["Selected Ult CC"] = e1_cc_ult.latest_diagonal.to_frame()
e1_s3[['Closed CDF','Reported CDF']] = e1_s3[['Closed CDF','Reported CDF']].round(3)
#using floating point offset to achieve standard rounding
e1_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] = (e1_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] + 1e-9).round(0)
e1_s3[['development','Closed Claim Counts','Reported Claim Counts','Closed CDF','Reported CDF','Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']]

## Exhibit I Sheet 4

In [0]:
(e1_tri['Closed Claim Counts']/e1_tri['Reported Claim Counts']).round(3)

## Exhibit I Sheet 5

In [0]:
nb_display(e1_tri['Reported Claims']/1000)
nb_display(e1_rsev)

## Exhibit I Sheet 6

In [0]:
print('PART 1 - Data Triangle')
nb_display(e1_rsev)
print('PART 2 - Age-to-Age Factors')
nb_display(e1_rsev.age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factors')
nb_display(combine_ldf(e1_rsev_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
nb_display(e1_rsev_selected.ldf_)
print('CDF to Ultimate')
nb_display(e1_rsev_selected.cdf_.round(3))

## Exhibit I Sheet 7

In [0]:
e1_rsev_ult_df = cl.model_diagnostics(e1_rsev_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e1_s7 = e1_rsev_ult_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Reported Severity','Ultimate':'Ult Severity'})
e1_s7['Selected Ult CC'] = e1_s3['Selected Ult CC']
e1_s7["Ult Claims"] = e1_ult.latest_diagonal.to_frame()
e1_s7['CDF'] = e1_s7['CDF'].round(3)
#using floating point offset to achieve standard rounding
e1_s7[['Reported Severity','Ult Severity','Ult Claims']] = (e1_s7[['Reported Severity','Ult Severity','Ult Claims']]+1e-9).round(0)
e1_s7

## Exhibit I Sheet 8

In [0]:
summary_exh(e1_tri['Reported Claims']/1000,e1_tri['Paid Claims']/1000,e1_ult)

At the end of each exhibit, we reconcile to hardcoded figures from Friedland using a series of ``assert`` statemenets. When any of these statements errors out, we know some bug has been introduced into the package. 

In [0]:
#Exhibit I Sheet 1
assert np.all(e1_ccc_selected.cdf_.round(3).values == np.array([1.305, 1.010, 1.001, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000]))
#Exhibit I Sheet 2
assert np.all(e1_rcc_selected.cdf_.round(3).values == np.array([0.975, 0.997, 0.999, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000]))
#Exhibit I Sheet 3
assert np.allclose(
    e1_s3['Selected Ult CC'].values,
    np.array([3292, 3243, 2699, 2757, 2635, 2699, 2665, 2524, 2764, 3061]),
    atol=1
)
#Exhibit I Sheet 6
assert np.all(e1_rsev_selected.cdf_.round(3).values == np.array([1.036, 0.997, 0.998, 0.999, 1.000, 1.000, 1.000, 1.000, 1.000, 1.000]))
#Exhibit I Sheet 7
assert np.allclose(
    e1_s7['Ult Severity'].values,
    np.array([4510, 4507, 4632, 4295, 4465, 4356, 4754, 4535, 4613, 4644]),
    atol=1
)

## Exhibit II Analysis

Exhibit II also follows Approach 1, and is largely similar to Exhibit I. 

In [0]:
#loading data and assumptions
e2_tri = cl.load_sample('friedland_xyz_auto_bi')
e2_cnt_assumptions = {}
e2_cnt_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e2_cnt_assumptions['simple_2'] = {'n_periods':2, 'average':'simple'}
e2_cnt_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e2_cnt_assumptions['volume_3'] = {'n_periods':3, 'average':'volume'}
e2_cnt_assumptions['volume_2'] = {'n_periods':2, 'average':'volume'}
e2_sev_assumptions = {}
e2_sev_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e2_sev_assumptions['simple_2'] = {'n_periods':2, 'average':'simple'}
e2_sev_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}

#developing closed claim counts
e2_ccc_devs = average_dev(e2_tri['Closed Claim Counts'],e2_cnt_assumptions)
e2_ccc_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e2_ccc_devs['volume_2'])
e2_ccc_selected.ldf_ = e2_ccc_selected.ldf_.round(3)

#developing reported claim counts
e2_rcc_devs = average_dev(e2_tri['Reported Claim Counts'],e2_cnt_assumptions)
e2_rcc_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e2_rcc_devs['volume_2'])
e2_rcc_selected.ldf_ = e2_rcc_selected.ldf_.round(3)

#combining closed and reported claim counts
e2_ccc_cl = cl.Chainladder().fit(e2_ccc_selected)
e2_rcc_cl = cl.Chainladder().fit(e2_rcc_selected)
e2_cc_ult = (e2_ccc_cl.ultimate_ + e2_rcc_cl.ultimate_)/2

#calculating and developing reported severity
e2_rsev = e2_tri['Reported Claims'] / e2_tri['Reported Claim Counts'] * 1000
e2_rsev_devs = average_dev(e2_rsev,e2_sev_assumptions)
e2_rsev_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e2_rsev_devs['simple_2'])
e2_rsev_selected.ldf_ = e2_rsev_selected.ldf_.round(3)
e2_rsev_cl = cl.Chainladder().fit(e2_rsev_selected)

#combining developed count and severity
e2_ult = e2_rsev_cl.ultimate_ * e2_cc_ult / 1000

   
## Exhibit II Sheet 1

In [0]:
print('PART 1 - Data Triangle')
nb_display(e2_tri['Closed Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e2_tri['Closed Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e2_ccc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e2_ccc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e2_ccc_selected.cdf_.round(3))
print('Percent Closed')
nb_display(1/e2_ccc_selected.cdf_.round(3))

   
## Exhibit II Sheet 2

In [0]:
print('PART 1 - Data Triangle')
nb_display(e2_tri['Reported Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e2_tri['Reported Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e2_rcc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e2_rcc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e2_rcc_selected.cdf_.round(3))
print('Percent Reported')
nb_display(1/e2_rcc_selected.cdf_.round(3))

   
## Exhibit II Sheet 3

In [0]:
e2_ccc_df = cl.model_diagnostics(e2_ccc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e2_rcc_df = cl.model_diagnostics(e2_rcc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e2_s3 = e2_ccc_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Closed Claim Counts','CDF':'Closed CDF','Ultimate':'Ult Count Using CCC'})
e2_s3[['Reported Claim Counts','Reported CDF','Ult Count Using RCC']] = e2_rcc_df[['Latest','CDF','Ultimate']]
e2_s3["Selected Ult CC"] = e2_cc_ult.latest_diagonal.to_frame()
e2_s3[['Closed CDF','Reported CDF']] = e2_s3[['Closed CDF','Reported CDF']].round(3)
#using floating point offset to achieve standard rounding
e2_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] = (e2_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] + 1e-9).round(0)
e2_s3[['development','Closed Claim Counts','Reported Claim Counts','Closed CDF','Reported CDF','Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']]

   
## Exhibit II Sheet 4

In [0]:
nb_display(e2_tri['Reported Claims'])
nb_display(e2_rsev)

   
## Exhibit II Sheet 5

In [0]:
print('PART 1 - Data Triangle')
nb_display(e2_rsev)
print('PART 2 - Age-to-Age Factors')
nb_display(e2_rsev.age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factors')
nb_display(combine_ldf(e2_rsev_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
nb_display(e2_rsev_selected.ldf_)
print('CDF to Ultimate')
nb_display(e2_rsev_selected.cdf_.round(3))

   
## Exhibit II Sheet 6

In [0]:
e2_rsev_df = cl.model_diagnostics(e2_rsev_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e2_s6 = e2_rsev_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Reported Severity','Ultimate':'Ult Severity'})
e2_s6['Selected Ult CC'] = e2_s3['Selected Ult CC']
e2_s6["Ult Claims"] = e2_ult.latest_diagonal.to_frame()
e2_s6['CDF'] = e2_s6['CDF'].round(3)
#using floating point offset to achieve standard rounding
e2_s6[['Reported Severity','Ult Severity','Ult Claims']] = (e2_s6[['Reported Severity','Ult Severity','Ult Claims']]+1e-9).round(0)
e2_s6

   
## Exhibit II Sheet 7

In [0]:
summary_exh(e2_tri['Reported Claims'],e2_tri['Paid Claims'],e2_ult)

In [0]:
#Exhibit II Sheet 1
assert np.all(e2_ccc_selected.cdf_.round(3).values == np.array([6.085, 2.281, 1.612, 1.318, 1.161, 1.085, 1.035, 1.021, 1.015, 1.003, 1.000]))
#Exhibit II Sheet 2
assert np.all(e2_rcc_selected.cdf_.round(3).values == np.array([1.131, 1.035, 1.013, 1.005, 1.002, 1.001, 1.000, 1.000, 1.000, 1.000, 1.000]))
#Exhibit II Sheet 3
assert np.allclose(
    e2_s3['Selected Ult CC'].values,
    np.array([637, 1047, 1416, 1466, 1565, 1666, 2309, 2483, 1807, 1556, 1426]),
    atol=1
)
#Exhibit II Sheet 5
assert np.all(e2_rsev_selected.cdf_.round(3).values == np.array([2.310, 1.498, 1.212, 1.100, 1.066, 1.016, 1.003, 0.992, 0.990, 0.999, 1.000]))
#Exhibit II Sheet 6
assert np.allclose(
    e2_s6['Ult Severity'].values,
    np.array([24839, 23956, 26189, 26452, 31090, 27675, 33183, 32519, 35697, 37606, 41544]),
    rtol=0.001
)

## Exhibit III Analysis

Exhibit III follows Approach 2, which adds explicit selections for the most recent frequency and severity. To calculate various averages for frequency and severity, we use the ``TriangleWeight`` utility class, and supply ``n_periods`` and ``drop_high`` as we would with ``Development``. We also leverage the ``Trend`` adjustment method. 

In [0]:
#loading data and assumptions
e3_tri = cl.load_sample('friedland_wc_self_insurer')
e3_cnt_assumptions = {}
e3_cnt_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e3_cnt_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e3_cnt_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e3_cnt_assumptions['volume_5'] = {'n_periods':5, 'average':'volume'}
e3_cnt_assumptions['volume_3'] = {'n_periods':3, 'average':'volume'}
e3_sev_assumptions = {}
e3_sev_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e3_sev_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e3_sev_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e3_sevavg_assumptions = {}
e3_sevavg_assumptions['all_years'] = {}
e3_sevavg_assumptions['all_years_excl_hilo'] = {'drop_high':1, 'drop_low':1}
e3_sevavg_assumptions['latest_3'] = {'n_periods':3}

#developing closed claim counts
#There is a typo in the text. We will use a tail of 1.002 to match the 84-ult factor'
e3_ccc_devs = average_dev(e3_tri['Closed Claim Counts'],e3_cnt_assumptions)
e3_ccc_selected = cl.TailConstant(tail = 1.002, projection_period = 0).fit_transform(e3_ccc_devs['volume_5'])
e3_ccc_selected.ldf_ = e3_ccc_selected.ldf_.round(3)

#developing reported claim counts
e3_rcc_devs = average_dev(e3_tri['Reported Claim Counts'],e3_cnt_assumptions)
e3_rcc_selected = cl.TailConstant(tail = 1.000, projection_period = 0).fit_transform(e3_rcc_devs['volume_5'])
e3_rcc_selected.ldf_ = e3_rcc_selected.ldf_.round(3)

#combining closed and reported claim counts
e3_ccc_cl = cl.Chainladder().fit(e3_ccc_selected)
e3_rcc_cl = cl.Chainladder().fit(e3_rcc_selected)
e3_cc_ult = (e3_ccc_cl.ultimate_ + e3_rcc_cl.ultimate_)/2

#calculating trended frequency
e3_freq_trend = cl.Trend(-.01).fit(e3_tri['Reported Claim Counts']).trend_
e3_payroll_trend = cl.Trend(.025).fit(e3_tri['Payroll']).trend_
e3_freq_2008 = .0036
e3_freq_2007 = e3_freq_2008 / e3_freq_trend.values[0,0,-2,0] * e3_payroll_trend.latest_diagonal.values[0,0,-2,0]

#calculating and developing paid severity
e3_psev = e3_tri["Paid Claims"] / e3_tri['Closed Claim Counts']
e3_psev_devs = average_dev(e3_psev,e3_sev_assumptions)
e3_psev_selected = cl.TailConstant(tail = 1.150, projection_period = 0).fit_transform(e3_psev_devs['medial_5x1'])
e3_psev_selected.ldf_ = e3_psev_selected.ldf_.round(3)

#calculating and developing reported severity
e3_rsev = e3_tri["Reported Claims"] / e3_tri['Reported Claim Counts']
e3_rsev_devs = average_dev(e3_rsev,e3_sev_assumptions)
e3_rsev_selected = cl.TailConstant(tail = 1.025, projection_period = 0).fit_transform(e3_rsev_devs['medial_5x1'])
e3_rsev_selected.ldf_ = e3_rsev_selected.ldf_.round(3)

#combining developed count and severity
e3_psev_cl = cl.Chainladder().fit(e3_psev_selected)
e3_resv_cl = cl.Chainladder().fit(e3_rsev_selected)
e3_sev_ult = (e3_psev_cl.ultimate_ + e3_resv_cl.ultimate_)/2

#calculated trended severity
e3_sev_trend = cl.Trend(.075).fit(e3_tri['Reported Claims']).trend_
e3_sev_ult_trended = e3_sev_ult * e3_sev_trend.latest_diagonal.values
#prep ult severity for average calculations
e3_sev_ult_trended_2006 = e3_sev_ult_trended[e3_sev_ult_trended.origin<='2006']
e3_sevs = average_sev(e3_sev_ult_trended_2006,e3_sevavg_assumptions)
#calculate selected severities
e3_sev_2008 = 7100.
e3_sev_2007 = e3_sev_2008 / e3_sev_trend.values[0,0,-2,0]


## Exhibit III Sheet 1

In [0]:
print('PART 1 - Data Triangle')
nb_display(e3_tri["Closed Claim Counts"])
print('PART 2 - Age-to-Age Factors')
nb_display(e3_tri['Closed Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e3_ccc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e3_ccc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e3_ccc_selected.cdf_.round(3))
print('Percent Closed')
nb_display((1/e3_ccc_selected.cdf_).round(3))

## Exhibit III Sheet 2

In [0]:
print('PART 1 - Data Triangle')
nb_display(e3_tri["Reported Claim Counts"])
print('PART 2 - Age-to-Age Factors')
nb_display(e3_tri['Reported Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e3_rcc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
nb_display(e3_rcc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e3_rcc_selected.cdf_.round(3))
print('Percent Reported')
nb_display((1/e3_rcc_selected.cdf_).round(3))

## Exhibit III Sheet 3

In [0]:
e3_ccc_df = cl.model_diagnostics(e3_ccc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e3_rcc_df = cl.model_diagnostics(e3_rcc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e3_s3 = e3_ccc_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Closed Claim Counts','CDF':'Closed CDF','Ultimate':'Ult Count Using CCC'})
e3_s3[['Reported Claim Counts','Reported CDF','Ult Count Using RCC']] = e3_rcc_df[['Latest','CDF','Ultimate']]
e3_s3["Selected Ult CC"] = e3_cc_ult.latest_diagonal.to_frame()
e3_s3[['Closed CDF','Reported CDF']] = e3_s3[['Closed CDF','Reported CDF']].round(3)
#using floating point offset to achieve standard rounding
e3_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] = (e3_s3[['Ult Count Using CCC','Ult Count Using RCC','Selected Ult CC']] + 1e-9).round(0)
e3_s3

## Exhibit III Sheet 4

In [0]:
e3_s4 = e3_s3[['Selected Ult CC']].copy()
e3_s4['CC Trend'] = e3_freq_trend.latest_diagonal.to_frame()
e3_s4['Trended Ult CC'] = e3_s4['Selected Ult CC'] * e3_s4['CC Trend']
e3_s4['Payroll'] = e3_tri['Payroll'].latest_diagonal.to_frame()
e3_s4['Payroll Trend'] = e3_payroll_trend.latest_diagonal.to_frame()
e3_s4['Trended Payroll'] = e3_s4['Payroll'] * e3_s4['Payroll Trend']
e3_s4['Trended Ultimate Freq'] = e3_s4['Trended Ult CC'] / e3_s4['Trended Payroll']
e3_s4[['CC Trend','Payroll Trend']] = e3_s4[['CC Trend','Payroll Trend']].round(3)
#using floating point offset to achieve standard rounding
e3_s4[['Trended Ult CC','Trended Payroll']] = (e3_s4[['Trended Ult CC','Trended Payroll']] + 1e-9).round(0)
e3_s4['Trended Ultimate Freq'] = e3_s4["Trended Ultimate Freq"].map("{:.2%}".format)
nb_display(e3_s4)
print(f"Selected Frequency at 2008 level {e3_freq_2008:.2%}")
print(f"Selected Frequency at 2007 level {float(e3_freq_2007):.2%}")

## Exhibit III Sheet 5

In [0]:
print('Paid Claims')
nb_display(e3_tri["Paid Claims"])
print('Paid Severities')
nb_display(e3_psev)
print('Reported Claims')
nb_display(e3_tri["Reported Claims"])
print('Reported Severities')
nb_display(e3_rsev)

## Exhibit III Sheet 6

In [0]:
print('PART 1 - Data Triangle')
nb_display(e3_psev)
print('PART 2 - Age-to-Age Factors')
nb_display(e3_psev.age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e3_psev_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e3_psev_selected.ldf_)
print('CDF to Ultimate')
nb_display(e3_psev_selected.cdf_.round(3))

## Exhibit III Sheet 7

In [0]:
print('PART 1 - Data Triangle')
nb_display(e3_rsev)
print('PART 2 - Age-to-Age Factors')
nb_display(e3_rsev.age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e3_rsev_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e3_rsev_selected.ldf_)
print('CDF to Ultimate')
nb_display(e3_rsev_selected.cdf_.round(3))

## Exhibit III Sheet 8

In [0]:
e3_psev_df = cl.model_diagnostics(e3_psev_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e3_resv_df = cl.model_diagnostics(e3_resv_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e3_s8 = e3_psev_df[['development','Latest','CDF','Ultimate']].rename(columns={'Latest':'Paid Sev','CDF':'Paid CDF','Ultimate':'Ult Paid Sev'})
e3_s8[['Reported Sev','Reported CDF','Ult Reported Sev']] = e3_resv_df[['Latest','CDF','Ultimate']]
e3_s8["Selected Ult Sev"] = e3_sev_ult.latest_diagonal.to_frame()
e3_s8[['Paid CDF','Reported CDF']] = e3_s8[['Paid CDF','Reported CDF']].round(3)
#using floating point offset to achieve standard rounding
e3_s8[['Paid Sev','Reported Sev','Ult Paid Sev','Ult Reported Sev','Selected Ult Sev']] = (e3_s8[['Paid Sev','Reported Sev','Ult Paid Sev','Ult Reported Sev','Selected Ult Sev']] + 1e-9).round(0)
e3_s8.iloc[:-2,:][['development','Paid Sev','Reported Sev','Paid CDF','Reported CDF','Ult Paid Sev','Ult Reported Sev','Selected Ult Sev']]

## Exhibit III Sheet 9

In [0]:
e3_s9 = e3_s8[['Selected Ult Sev']].copy()
e3_s9['Sev Trend'] = e3_sev_trend.latest_diagonal.to_frame()
e3_s9['Trended Ult Sev'] = e3_sev_ult_trended.latest_diagonal.to_frame()
e3_s9[['Sev Trend']] = e3_s9[['Sev Trend']].round(3)
#using floating point offset to achieve standard rounding
e3_s9[['Trended Ult Sev']] = (e3_s9[['Trended Ult Sev']] + 1e-9).round(0)
nb_display(e3_s9.iloc[:-2,:])
print('Average Trended Severity at 2008 Cost Level')
for k,v in e3_sevs.items():
    print(f"\t{k:<20} \t{v.round(0)}")
print(f"Selected 2008 Severity \t\t{e3_sev_2008:.0f}")
print(f"Estimated 2007 Severity \t{e3_sev_2007:.0f}")

## Exhibit III Sheet 10

In [0]:
e3_s10 = e3_s4[['Payroll']].copy()[-2:]
e3_s10['Selected Frequency'] = [e3_freq_2007,e3_freq_2008]
e3_s10['Ult CC'] = e3_s10['Payroll'] * e3_s10['Selected Frequency']
e3_s10['Selected Severity'] = [e3_sev_2007,e3_sev_2008]
e3_s10['Ult Claims'] = e3_s10['Ult CC'] * e3_s10['Selected Severity']
e3_s10['Reported Claims'] = e3_tri['Reported Claims'].latest_diagonal.to_frame()
e3_s10['Paid Claims'] = e3_tri['Paid Claims'].latest_diagonal.to_frame()
e3_s10['Case Outstanding'] = e3_s10['Reported Claims'] - e3_s10['Paid Claims']
e3_s10['IBNR'] = e3_s10['Ult Claims'] - e3_s10['Reported Claims']
e3_s10['Unpaid'] = e3_s10['IBNR'] + e3_s10['Case Outstanding']
e3_s10.T

In [0]:
#Exhibit III Sheet 1
assert np.all(e3_ccc_selected.cdf_.round(3).values[...,:7] == np.array([1.698, 1.133, 1.071, 1.034, 1.018, 1.014, 1.010]))
#Exhibit III Sheet 2
assert np.all(e3_rcc_selected.cdf_.round(3).values == np.array([1.094, 1.022, 1.007, 1.002, 1.000, 1.000, 1.000, 1.000]))
#Exhibit III Sheet 3
assert np.allclose(
    e3_s3['Selected Ult CC'].values[...,1:],
    np.array([1700, 1774, 1749, 1651, 2982, 2909, 2658]),
    atol=1
)
#Exhibit III Sheet 4
assert np.all(e3_s4['Trended Ultimate Freq'].values[...,1:] == np.array(['0.53%', '0.53%', '0.54%', '0.43%', '0.35%', '0.36%', '0.36%']))
#Exhibit III Sheet 6
assert np.allclose(
    e3_psev_selected.ldf_.round(3).values,
    np.array([1.447, 1.249, 1.108, 1.066, 1.038, 1.037, 1.021, 1.150]),
    atol=.001
)
#Exhibit III Sheet 7
assert np.all(e3_rsev_selected.cdf_.round(3).values == np.array([1.679, 1.314, 1.189, 1.130, 1.092, 1.065, 1.043, 1.025]))
#Exhibit III Sheet 8
assert np.allclose(
    e3_s8['Selected Ult Sev'].values[:-2],
    np.array([4371, 4587, 4963, 5242, 5635, 6169]),
    atol=1
)
#Exhibit III Sheet 9
assert np.allclose(
    e3_s9['Trended Ult Sev'].values[:-2],
    np.array([7251, 7079, 7125, 7000, 7001, 7129]),
    rtol=0.001
)

## Exhibit IV Analysis

Exhibit IV continues to follow Approach 2. We demonstrate two separate ways in the package to achievement the on-leveling and law-change adjustments. Also, there are no full-size triangles in this exhibit. This gives us an opportunity to demontrate utilizing `columns` in `Triangle`, as if we were in Pandas.

In [0]:
#loading data and assumptions
xyz_tort_data = pd.DataFrame({
    "EffDate": ["2006-01-01", "2007-01-01"],
    "RateChange": [(0.67/0.75) - 1, 0.75-1]
})
xyz_tort_adjustment = cl.ParallelogramOLF(
    rate_history=xyz_tort_data,
    change_col="RateChange",
    date_col="EffDate",
    vertical_line=True
)
#Both Chapter 6 and Chapter 8 contain the code to derive these on-level factors from underlying rate changes. We will simply use the factors as given
olf = cl.Triangle(
    data = {
        'origin':[1998,2002,2003,2004,2005,2006,2007,2008],
        'valuation':[2008,2008,2008,2008,2008,2008,2008,2008],
        'On-Level Adjustment':[0,.914,.87,.81,.704,.64,.8,1],
    },
    origin='origin',
    development='valuation',
    columns='On-Level Adjustment'
)
e4_freq_assumptions = {}
e4_freq_assumptions['all_years'] = {}
e4_freq_assumptions['all_years_excl_hilo'] = {'drop_high':1, 'drop_low':1}
e4_freq_assumptions['latest_2'] = {'n_periods':2}
e4_sev_assumptions = {}
e4_sev_assumptions['latest_5_years'] = {'n_periods':5}
e4_sev_assumptions['latest_5_years_excl_hilo'] = {'n_periods':5,'drop_high':1, 'drop_low':1}
e4_sev_assumptions['latest_3'] = {'n_periods':3}

# In Exhibit II, the selected ultimate was actually the average of reported and closed ultimates.
e4_s1_tri = e2_rcc_cl.ultimate_.copy()
e4_s1_tri = e4_s1_tri.rename(axis='columns',value='Ult CC')
e4_s1_tri['CC Trend'] = cl.Trend(-.015,dates=('2008-12-31','2002-01-01')).fit(e4_s1_tri).trend_
e4_s1_tri['Trended Ult CC'] = e4_s1_tri['CC Trend'] * e4_s1_tri['Ult CC']
e4_s1_tri['Earned Premium'] = e2_tri['Earned Premium'].latest_diagonal
e4_s1_tri['On-Level Adjustment'] = olf['On-Level Adjustment']
e4_s1_tri['On-Level Premium'] = e4_s1_tri['On-Level Adjustment'] * e4_s1_tri['Earned Premium']
e4_s1_tri['Trended Ult Freq'] = e4_s1_tri['Trended Ult CC'] / e4_s1_tri['On-Level Premium']
e4_s1_tri_2006 = e4_s1_tri.iloc[:,:,4:9,:]
#calculate average frequencies
e4_freq = average_sev(e4_s1_tri_2006['Trended Ult Freq'],e4_freq_assumptions)
#calculate selected frequencies
e4_freq_2008 = e4_freq['latest_2'].round(4)
e4_freq_2007 = e4_freq_2008 / e4_s1_tri['CC Trend'].values[0,0,-2,0] * e4_s1_tri['On-Level Adjustment'].values[0,0,-2,0]
e4_freq_2007 = e4_freq_2007.round(4)

e4_s2_tri = e2_rsev_cl.ultimate_.copy()
e4_s2_tri = e4_s2_tri.rename(axis='columns',value='Ult Sev')
e4_s2_tri['Sev Trend'] = cl.Trend(.05,dates=('2008-12-31','1998-01-01')).fit(e4_s2_tri).trend_.round(3)
#from Chapter 8 Exhibit III sheet 1
e4_s2_tri['Tort Reform Factors'] = xyz_tort_adjustment.fit(e4_s2_tri['Ult Sev']).olf_
e4_s2_tri['Trended Ult Sev'] = e4_s2_tri['Ult Sev'] * e4_s2_tri['Sev Trend'] * e4_s2_tri['Tort Reform Factors']
e4_s2_tri_2006 = e4_s2_tri[e4_s2_tri.origin <= '2006']
#calculate average severities
e4_sev = average_sev(e4_s2_tri_2006['Trended Ult Sev'],e4_sev_assumptions)
#calculate selected severities
e4_sev_2008 = e4_sev['latest_5_years_excl_hilo'].round(0)
e4_sev_2007 = e4_sev_2008 / e4_s2_tri['Sev Trend'].values[0,0,-2,0] / e4_s2_tri['Tort Reform Factors'].values[0,0,-2,0]
e4_sev_2007 = e4_sev_2007.round(0)

## Exhibit IV Sheet 1

In [0]:
e4_s1 = e4_s1_tri.to_frame().T
e4_s1[['CC Trend','On-Level Adjustment']] = e4_s1[['CC Trend','On-Level Adjustment']].round(3)
#using floating point offset to achieve standard rounding
e4_s1[['Trended Ult CC','On-Level Premium']] = (e4_s1[['Trended Ult CC','On-Level Premium']] + 1e-9).round(0)
e4_s1['Trended Ult Freq'] = e4_s1["Trended Ult Freq"].map("{:.2%}".format)
nb_display(e4_s1.iloc[4:9])
print('Average Trended Frequency at 2008 Cost Level')
for k,v in e4_freq.items():
    print(f"\t{k:<20} \t{v:.2%}")
print(f"Selected 2008 Frequency \t{e4_freq_2008:.2%}")
print(f"Estimated 2007 Frequency \t{e4_freq_2007:.2%}")

## Exhibit IV Sheet 2

In [0]:
e4_s2 = e4_s2_tri.to_frame().T
e4_s2[['Sev Trend','Tort Reform Factors']] = e4_s2[['Sev Trend','Tort Reform Factors']].round(3)
#using floating point offset to achieve standard rounding
e4_s2[['Trended Ult Sev','Ult Sev']] = (e4_s2[['Trended Ult Sev','Ult Sev']] + 1e-9).round(0)
nb_display(e4_s2.iloc[:9])
print('Average Trended Severity at 2008 Cost Level')
for k,v in e4_sev.items():
    print(f"\t{k:<25} \t{v.round(0)}")
print(f"Selected 2008 Severity \t\t\t{e4_sev_2008}")
print(f"Estimated 2007 Severity \t\t{e4_sev_2007}")

## Exhibit IV Sheet 3

In [0]:
e4_s3 = e4_s1_tri['Earned Premium'].to_frame(keepdims=True).set_index('origin').iloc[-2:][['Earned Premium']]
e4_s3['Selected Frequency'] = [e4_freq_2007,e4_freq_2008]
e4_s3['Ult CC'] = e4_s3['Earned Premium'] * e4_s3['Selected Frequency']
e4_s3['Selected Severity'] = [e4_sev_2007,e4_sev_2008]
e4_s3['Ult Claims'] = e4_s3['Ult CC'] * e4_s3['Selected Severity']
e4_s3['Reported Claims'] = (e2_tri['Reported Claims']*1000).latest_diagonal.to_frame()
e4_s3['Paid Claims'] = (e2_tri['Paid Claims']*1000).latest_diagonal.to_frame()
e4_s3['Case Outstanding'] = e4_s3['Reported Claims'] - e4_s3['Paid Claims']
e4_s3['IBNR'] = e4_s3['Ult Claims'] - e4_s3['Reported Claims']
e4_s3['Unpaid'] = e4_s3['IBNR'] + e4_s3['Case Outstanding']
e4_s3.T

In [0]:
#Exhibit IV Sheet 1
assert np.all(e4_s1.iloc[4:9]['Trended Ult Freq'].values == np.array(['2.54%', '2.51%', '2.65%', '2.36%', '2.37%']))
#Exhibit IV Sheet 2
assert np.allclose(
    list(e4_sev.values()),
    np.array([26669., 26720., 27254.]),
    rtol=0.001
)
#Exhibit IV Sheet 3
assert np.allclose(
    e4_s3.iloc[:,4].values,
    np.array([30512152., 30140260.]),
    rtol=0.001
)

## Exhibit V Analysis

Exhibit V follows Approach 3, which uses full incremental closed count and paid triangles to estimate the ultimate. Full incremental closed count triangle is estimated after first estimating ultimate count using reported count. We use the ``DisposalRate`` adjustment method to perform this calculation. Full incremental paid severity triangle is estimated through trending. 

In [0]:
#loading data and assumptions
e5_tri = cl.load_sample('friedland_gl_insurer')
e5_cnt_assumptions = {}
e5_cnt_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e5_cnt_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e5_cnt_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e5_cnt_assumptions['volume_5'] = {'n_periods':5, 'average':'volume'}
e5_cnt_assumptions['volume_3'] = {'n_periods':3, 'average':'volume'}
e5_disp_assumptions = {}
e5_disp_assumptions['simple_5'] = {'n_periods':5, 'average':'simple'}
e5_disp_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e5_disp_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e5_trend_assumptions = {}
e5_trend_assumptions['all_years'] = {}
e5_trend_assumptions['latest_6'] = {'n_periods':6}
e5_trend_assumptions['latest_4'] = {'n_periods':4}
e5_sev_assumptions = {}
e5_sev_assumptions['simple_5'] = {'n_periods':5}
e5_sev_assumptions['simple_3'] = {'n_periods':3}
e5_sev_assumptions['medial_5x1'] = {'n_periods':5,'drop_high':1, 'drop_low':1}

#developing closed claim counts
e5_ccc_devs = average_dev(e5_tri['Closed Claim Counts'],e5_cnt_assumptions)
e5_ccc_selected = cl.TailConstant(tail = 1.100, projection_period = 0).fit_transform(e5_ccc_devs['volume_3'])
e5_ccc_selected.ldf_ = e5_ccc_selected.ldf_.round(3)

#developing reported claim counts
e5_rcc_devs = average_dev(e5_tri['Reported Claim Counts'],e5_cnt_assumptions)
e5_rcc_selected = cl.TailConstant(tail = 1.0, projection_period = 0).fit_transform(e5_rcc_devs['volume_3'])
e5_rcc_selected.ldf_ = e5_rcc_selected.ldf_.round(3)

#combining closed and reported claim counts
e5_ccc_cl = cl.Chainladder().fit(e5_ccc_selected)
e5_rcc_cl = cl.Chainladder().fit(e5_rcc_selected)
e5_cc_ult = ((e5_ccc_cl.ultimate_ + e5_rcc_cl.ultimate_)/2)

#calculating disposal rate and complete the closed count triangle
e5_drs = average_dr(e5_tri['Closed Claim Counts'],e5_cc_ult,e5_disp_assumptions)
e5_dr_selected = e5_drs['medial_5x1']
e5_dr_selected.disposal_rate_ = e5_dr_selected.disposal_rate_.round(3)
e5_dr_tri = e5_dr_selected.transform(e5_tri['Closed Claim Counts'],sample_weight = e5_cc_ult)

#calculate and select incremental paid severity
e5_ipsev = e5_tri["Paid Claims"].cum_to_incr() / e5_tri["Closed Claim Counts"].cum_to_incr()
e5_regs = regs(e5_ipsev,e5_trend_assumptions)
e5_trend,e5_rsq = reg_outputs(e5_regs,e5_ipsev.development)
e5_regs_ex2001 = regs(e5_ipsev[e5_ipsev.origin>'2001'],{'all_years_ex_2001':{}})
e5_trend_ex2001,e5_rsq_ex2001 = reg_outputs(e5_regs_ex2001,e5_ipsev.development)
e5_sevs = average_sev(e5_ipsev.trend(0.05),e5_sev_assumptions)
e5_sevs_sel = e5_sevs['simple_3'].copy()
e5_sevs_sel.origin = ['2008']

#calculate and select tail paid severity
e5_incr_ccc = e5_tri["Closed Claim Counts"].cum_to_incr()
e5_incr_paid = e5_tri["Paid Claims"].cum_to_incr()
e5_incr_paid_trended = e5_incr_paid.trend(0.05)

#complete the incremental paid severity triangle
e5_ipsev_full = e5_ipsev.copy()
#extending the severity triangle to 108 months
e5_ipsev_full = cl.concat((e5_ipsev_full,e5_ipsev_full.latest_diagonal.rename("development",[9999])),axis=3)
#setting the before 60 months to selected incremental
e5_ipsev_full.iloc[:,:,:,:5] = e5_sevs_sel.iloc[:,:,:,:5]
#setting the after 72 months to selected tail
e5_ipsev_full.iloc[:,:,:,5:] = (
    e5_incr_paid_trended[e5_incr_paid_trended.development >= 72].sum().sum()
    /
    e5_incr_ccc[e5_incr_ccc.development >= 72].sum().sum() 
)
#setting the valuation_date to the future
e5_ipsev_full.valuation_date = pd.to_datetime(cl.options.ULT_VAL)
#trending back from latest accident year
e5_ipsev_full_detrended = e5_ipsev_full.trend(1/1.05-1,start='2008-12-31',end='2001-01-01')
#compositing the actual incremental severities with dtrended selected to complete the incremental paid severity triangle
e5_ipsev_full_complete = e5_ipsev[e5_ipsev.valuation <= e5_ipsev.valuation_date] + e5_ipsev_full_detrended[e5_ipsev_full_detrended.valuation > e5_ipsev.valuation_date]

#complete the incremental paid triangle
e5_s11_ccc = e5_dr_tri.full_triangle_.cum_to_incr()
e5_s11_ip = e5_s11_ccc * e5_ipsev_full_complete


## Exhibit V Sheet 1

In [0]:
print('PART 1 - Data Triangle')
nb_display(e5_tri['Closed Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e5_tri['Closed Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e5_ccc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
print('Selected')
nb_display(e5_ccc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e5_ccc_selected.cdf_.round(3))
print('Percent Closed')
nb_display((1/e5_ccc_selected.cdf_).round(3))

## Exhibit V Sheet 2

In [0]:
print('PART 1 - Data Triangle')
nb_display(e5_tri['Reported Claim Counts'])
print('PART 2 - Age-to-Age Factors')
nb_display(e5_tri['Reported Claim Counts'].age_to_age.round(3))
print('PART 3 - Average Age-to-Age Factor')
nb_display(combine_ldf(e5_rcc_devs).round(3).to_frame())
print('PART 4 - Selected Age-to-Age Factors')
nb_display(e5_rcc_selected.ldf_)
print('CDF to Ultimate')
nb_display(e5_rcc_selected.cdf_.round(3))
print('Percent Reported')
nb_display((1/e5_rcc_selected.cdf_).round(3))

## Exhibit V Sheet 3

In [0]:
e5_ccc_df = cl.model_diagnostics(e5_ccc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e5_rcc_df = cl.model_diagnostics(e5_rcc_cl).to_frame(keepdims=True,implicit_axis=True).set_index('origin')
e5_s3 = e5_ccc_df[['development','Latest','Ultimate']].rename(columns={'Latest':'Closed Claim Counts','Ultimate':'Ult Count Using CCC'})
e5_s3[['Reported Claim Counts','Ult Count Using RCC']] = e5_rcc_df[['Latest','Ultimate']]
e5_s3["Selected Ult CC"] = e5_cc_ult.round(0).latest_diagonal.to_frame()
e5_s3

## Exhibit V Sheet 4

In [0]:
print('Part 1 - Disposal Rate Triangle')
nb_display(cl.DisposalRate().fit_transform(e5_tri['Closed Claim Counts'],sample_weight = e5_cc_ult).disposal_rate_tri.round(3))
print('PART 2 - Average Disposal Rate Factors')
nb_display(combine_disposal(e5_drs).round(3).to_frame())
print('PART 3 - Selected Disposal Rate Factors')
nb_display(e5_dr_selected.disposal_rate_)

## Exhibit V Sheet 5

In [0]:
print('Closed Claim Counts')
nb_display(e5_tri['Closed Claim Counts'])
print('Projected Incremental Closed Claim Counts')
nb_display(e5_dr_tri.full_triangle_.cum_to_incr().round(0))

## Exhibit V Sheet 6

In [0]:
print('Paid Claims')
nb_display(e5_tri["Paid Claims"])
print('Incremental Paid CLaims')
nb_display(e5_tri["Paid Claims"].cum_to_incr())
print('Incremental Closed Claim Counts')
nb_display(e5_tri["Closed Claim Counts"].cum_to_incr())
print('Incremental Paid Severities')
nb_display(e5_ipsev)

## Exhibit V Sheet 7

In [0]:
print('Incremental Paid Severities')
nb_display(e5_ipsev)
print('Annual Change based on Exponential Regression')
nb_display(pd.concat([e5_trend.round(3),e5_trend_ex2001.round(3)]))
print('Goodness of Fit Test of Exponential Regression (R-Squared)')
nb_display(pd.concat([e5_rsq.round(3),e5_rsq_ex2001.round(3)]))

## Exhibit V Sheet 8

In [0]:
print('Incremental Paid Severities')
nb_display(e5_ipsev)
print('Trended Incremental Paid Severities Assuming 5% Annual Trend')
nb_display(e5_ipsev.trend(0.05))
print('Average Trended Incremental Paid Severities')
nb_display(combine_tri(e5_sevs).round(0).to_frame())
print('Selected Incremental Paid Severities')
nb_display(e5_sevs_sel[e5_sevs_sel.development<=60])

## Exhibit V Sheet 9

In [0]:
e5_s9 = pd.DataFrame(
    data = [[
        e5_incr_ccc[e5_incr_ccc.development >= 60].sum().sum(), 
        e5_incr_ccc[e5_incr_ccc.development >= 72].sum().sum()
    ]],
    index = ['Total Closed Claim Counts'],
    columns = ['Age 60 & Older','Age 72 & Older']
)
e5_s9.loc['Total Trended Paid Claims'] = [
        e5_incr_paid_trended[e5_incr_paid_trended.development >= 60].sum().sum(), 
        e5_incr_paid_trended[e5_incr_paid_trended.development >= 72].sum().sum()
    ]
e5_s9.loc['Estimated Trended Tail Severity'] = e5_s9.iloc[1] / e5_s9.iloc[0]
e5_s9.loc['Estimated Incremental Trended Tail Severity'] = e5_sevs_sel.iloc[:,:,:,4:6].values.flatten()
e5_s9.loc['Selected Trended Paid Severity'] = e5_ipsev_full.values[0,0,-1,4:6]
print('Incremental Closed Claim Counts')
nb_display(e5_incr_ccc[e5_incr_ccc.development >= 60][e5_incr_ccc.origin <= '2004'])
print('Incremental Paid CLaims')
nb_display(e5_incr_paid[e5_incr_paid.development >= 60][e5_incr_paid.origin <= '2004'])
print('Trended Incremental Paid Severities')
nb_display(e5_incr_paid_trended[e5_incr_paid_trended.development >= 60][e5_incr_paid_trended.origin <= '2004'])
nb_display(e5_s9)

## Exhibit V Sheet 10

In [0]:
print('Incremental Paid Severities')
nb_display(e5_ipsev)
print('Selected Incremental Paid Severities')
nb_display(e5_ipsev_full[e5_ipsev_full.origin == '2008'])
print('Incremental Paid Severities Adjusted to Cost Level of Accident Year Assuming 5% Annual Trend Rate')
nb_display(e5_ipsev_full_complete)

## Exhibit V Sheet 11

In [0]:
print('Projected Incremental Closed Claim Counts')
nb_display(e5_s11_ccc.round(0))
print('Incremental Paid Severities Adjusted to Cost Level of Accident Year Assuming 5% Annual Trend Rate')
nb_display(e5_ipsev_full_complete)
print('Projected Incremental Paid Claims')
nb_display(e5_s11_ip)
print('Projected Cumulative Paid Claims')
nb_display(e5_s11_ip.incr_to_cum())

## Exhibit V Sheet 12

In [0]:
summary_exh(
    e5_tri['Reported Claims']/1000,
    e5_tri['Paid Claims']/1000,
    e5_s11_ip.incr_to_cum()/1000
)

In [0]:
e5_dr_selected.disposal_rate_

In [0]:
#Exhibit V Sheet 1
assert np.all(e5_ccc_selected.cdf_.round(3).values == np.array([4.769, 2.199, 1.682, 1.390, 1.256, 1.174, 1.123, 1.100]))
#Exhibit V Sheet 2
assert np.all(e5_rcc_selected.cdf_.round(3).values == np.array([0.753, 0.746, 0.818, 0.897, 0.932, 0.968, 1.007, 1.000]))
#Exhibit V Sheet 3
assert np.all(e5_s3['Selected Ult CC'].values==np.array([873., 720., 626., 629., 588., 553, 438., 609.]))
#Exhibit V Sheet 4
assert np.all(e5_dr_selected.disposal_rate_.values == np.array([0.200, 0.433, 0.585, 0.710, 0.791, 0.862, 0.882, 0.912, 1.000]))
#Exhibit V Sheet 5
lhs = (e5_dr_tri.full_triangle_.cum_to_incr()-e5_tri['Closed Claim Counts'].cum_to_incr()).values.flatten()
rhs = np.array([
                                                        77.,  
                                                24.,    70.,  
                                        12.,    18.,    54.,  
                                46.,    13.,    19.,    57.,  
                        52.,    45.,    13.,    19.,    56.,  
                76.,    49.,    43.,    12.,    18.,    54.,  
        67.,    55.,    36.,    31.,    9.,     13.,    39.,  
140.,   91.,    75.,    49.,    43.,    12.,    18.,    53.
])
assert np.all(abs(lhs[~np.isnan(lhs)] - rhs) < 1)
#Exhibit V Sheet 7
assert np.all(e5_trend.loc['latest_4'].values[:7].round(3) == np.array([0.018, 0.016, 0.095, 0.050, 0.122, -0.294, -0.331]))
assert np.all(e5_rsq.loc['latest_6'].values[:7].round(3) == np.array([0.704, 0.000, 0.644, 0.722, 0.084, 0.190, 1.000]))
#Exhibit V Sheet 8
assert np.all(e5_sevs_sel.values[...,:5].round(0) == np.array([11259.,  32980.,  65523.,  80544., 140802.]))
#Exhibit V Sheet 9
assert np.all(e5_s9.iloc[2].values.round(0) == np.array([144160., 175816.]))
#Exhibit V Sheet 10
lhs = e5_ipsev_full_detrended[e5_ipsev_full_detrended.valuation > e5_ipsev.valuation_date].round(0).values.flatten()
rhs = np.array([
                                                        124949.,
                                                131196.,131196.,
                                        137756.,137756.,137756.,
                                144644.,144644.,144644.,144644.,
                        121630.,151876.,151876.,151876.,151876.,
                73056.,	127711.,159470.,159470.,159470.,159470.,
        62403.,	76709.,	134097.,167443.,167443.,167443.,167443.,
32980., 65523.,	80544.,	140802.,175816.,175816.,175816.,175816.,
])
assert np.all(lhs[~np.isnan(lhs)] == rhs)
#Exhibit V Sheet 11
assert np.allclose(
    e5_s11_ip.incr_to_cum().iloc[:,:,:,-1].values.round(0).flatten(),
    np.array([39497433., 44743308., 35510981., 37766027., 42007442., 41113459., 32859080., 47109641.]),
    rtol = .001
)

## Exhibit VI Analysis

In [0]:
#loading data and assumptions
e6_tri = e2_tri[e2_tri.origin >= '2001'][e2_tri.development <= 96]
e6_disp_assumptions = {}
e6_disp_assumptions['simple_3'] = {'n_periods':3, 'average':'simple'}
e6_disp_assumptions['simple_2'] = {'n_periods':2, 'average':'simple'}
e6_disp_assumptions['medial_5x1'] = {'n_periods':5, 'average':'simple','drop_high':1, 'drop_low':1}
e6_sev_assumptions = {}
e6_sev_assumptions['simple_3'] = {'n_periods':3}
e6_sev_assumptions['simple_2'] = {'n_periods':2}
e6_sev_assumptions['medial_5x1'] = {'n_periods':5,'drop_high':1, 'drop_low':1}

#calculating disposal rate and complete the closed count triangle
e6_rcc_ult = e2_rcc_cl.ultimate_[e2_rcc_cl.ultimate_.origin >= '2001']
e6_drs = average_dr(e6_tri['Closed Claim Counts'],e6_rcc_ult,e6_disp_assumptions)
e6_dr_selected = e6_drs['simple_2']
e6_dr_selected.disposal_rate_ = e6_dr_selected.disposal_rate_.round(3)
e6_dr_tri = e6_dr_selected.transform(e6_tri['Closed Claim Counts'],sample_weight = e6_rcc_ult)

#calculate and select incremental paid severity
e6_ipsev = e6_tri["Paid Claims"].cum_to_incr() / e6_tri["Closed Claim Counts"].cum_to_incr() * 1000
e6_ipsev_adj = e6_ipsev.trend(0.05) * xyz_tort_adjustment.fit(e6_ipsev).olf_.values
e6_sevs = average_sev(e6_ipsev_adj,e6_sev_assumptions)
e6_sevs_sel = e6_sevs['simple_2'].copy()
e6_sevs_sel.origin = ['2008']

#calculate and select tail paid severity
e6_incr_ccc = e6_tri["Closed Claim Counts"].cum_to_incr()
e6_incr_paid = e6_tri["Paid Claims"].cum_to_incr() * 1000
e6_incr_paid_trended = e6_ipsev_adj * e6_incr_ccc

e6_ipsev_full = e6_ipsev_adj.copy()
#extending the severity triangle to 108 months
e6_ipsev_full = cl.concat((e6_ipsev_full,e6_ipsev_full.latest_diagonal.rename("development",[9999])),axis=3)
#setting the before 72 months to selected incremental
e6_ipsev_full.iloc[:,:,:,:6] = e6_sevs_sel.iloc[:,:,:,:6]
#setting the after 86 months to selected tail
e6_ipsev_full.iloc[:,:,:,6:] = (
    e6_incr_paid_trended[e6_incr_paid_trended.development >= 84].sum().sum()
    /
    e6_incr_ccc[e6_incr_ccc.development >= 84].sum().sum() 
)
#setting the valuation_date to the future
e6_ipsev_full.valuation_date = pd.to_datetime(cl.options.ULT_VAL)
#trending back from latest accident year
e6_ipsev_full_detrended = e6_ipsev_full.trend(1/1.05-1,start='2008-12-31',end='2001-01-01') / xyz_tort_adjustment.fit(e6_ipsev).olf_.values
#compositing the actual incremental severities with dtrended selected
e6_ipsev_full_complete = e6_ipsev[e6_ipsev.valuation <= e6_ipsev.valuation_date] + e6_ipsev_full_detrended[e6_ipsev_full_detrended.valuation > e6_ipsev.valuation_date]

#complete the incremental paid triangle
e6_s7_ccc = e6_dr_tri.full_triangle_.cum_to_incr()
e6_s7_ip = e6_s7_ccc * e6_ipsev_full_complete / 1000


## Exhibit VI Sheet 1

In [0]:
print('Part 1 - Disposal Rate Triangle')
nb_display(cl.DisposalRate().fit_transform(e6_tri['Closed Claim Counts'],sample_weight = e6_rcc_ult).disposal_rate_tri.round(3))

print('PART 2 - Average Disposal Rate Factors')
nb_display(combine_disposal(e6_drs).round(3).to_frame())

print('PART 3 - Selected Disposal Rate Factors')
nb_display(e6_dr_selected.disposal_rate_)

## Exhibit VI Sheet 2

In [0]:
print('Closed Claim Counts')
nb_display(e6_tri['Closed Claim Counts'])
print('Projected Incremental Closed Claim Counts')
nb_display(e6_dr_tri.full_triangle_.cum_to_incr().round(0))

## Exhibit VI Sheet 3

In [0]:
print('Paid Claims')
nb_display(e6_tri["Paid Claims"])
print('Incremental Paid CLaims')
nb_display(e6_tri["Paid Claims"].cum_to_incr())
print('Incremental Closed Claim Counts')
nb_display(e6_tri["Closed Claim Counts"].cum_to_incr())
print('Incremental Paid Severities')
nb_display(e6_ipsev)

## Exhibit VI Sheet 4

In [0]:
print('Incremental Paid Severities')
nb_display(e6_ipsev)
print('Trended Incremental Paid Severities Assuming 5% Annual Trend and Adjusted for Tort Reform')
nb_display(e6_ipsev_adj)
print('Average Trended Incremental Paid Severities')
nb_display(combine_tri(e6_sevs).round(0).to_frame())
print('Selected Incremental Paid Severities')
nb_display(e6_sevs_sel[e6_sevs_sel.development<=72])

## Exhibit VI Sheet 5

In [0]:
print('Incremental Closed Claim Counts')
nb_display(e6_incr_ccc[e6_incr_ccc.development >= 72][e6_incr_ccc.origin <= '2003'])
print('Incremental Paid CLaims')
nb_display(e6_incr_paid[e6_incr_paid.development >= 72][e6_incr_paid.origin <= '2003'])
print('Trended Incremental Paid Severities')
nb_display(e6_incr_paid_trended[e6_incr_paid_trended.development >= 72][e6_incr_paid_trended.origin <= '2003'])
e6_s5 = pd.DataFrame(
    data = [[
        e6_incr_ccc[e6_incr_ccc.development >= 72].sum().sum(), 
        e6_incr_ccc[e6_incr_ccc.development >= 84].sum().sum()
    ]],
    index = ['Total Closed Claim Counts'],
    columns = ['Age 72 & Older','Age 84 & Older']
)
e6_s5.loc['Total Trended Paid Claims'] = [
        e6_incr_paid_trended[e6_incr_paid_trended.development >= 72].sum().sum(), 
        e6_incr_paid_trended[e6_incr_paid_trended.development >= 84].sum().sum()
    ]
e6_s5.loc['Estimated Trended Tail Severity'] = e6_s5.iloc[1] / e6_s5.iloc[0]
e6_s5.loc['Estimated Incremental Trended Tail Severity'] = e6_sevs_sel.iloc[:,:,:,5:7].values.flatten()
e6_s5.loc['Selected Trended Paid Severity'] = e6_ipsev_full.values[0,0,-1,5:7]
nb_display(e6_s5)

## Exhibit VI Sheet 6

In [0]:
print('Incremental Paid Severities')
nb_display(e6_ipsev)
print('Selected Incremental Paid Severities')
nb_display(e6_ipsev_full[e6_ipsev_full.origin == '2008'])
print('Incremental Paid Severities Adjusted to Cost Level of Accident Year Assuming 5% Annual Trend Rate and Tort Reform Adjustment')
nb_display(e6_ipsev_full_complete)

## Exhibit VI Sheet 7

In [0]:
print('Projected Incremental Closed Claim Counts')
nb_display(e6_s7_ccc.round(0))
print('Incremental Paid Severities Adjusted to Cost Level of Accident Year Assuming 5% Annual Trend Rate and Tort Reform Adjustment')
nb_display(e6_ipsev_full_complete)
print('Projected Incremental Paid Claims')
nb_display(e6_s7_ip)
print('Projected Cumulative Paid Claims')
nb_display(e6_s7_ip.incr_to_cum())

## Exhibit VI Sheet 8

In [0]:
summary_exh(
    e6_tri['Reported Claims'],
    e6_tri['Paid Claims'],
    e6_s7_ip.incr_to_cum()
)

In [0]:
#Exhibit VI Sheet 1
assert np.all(e6_dr_selected.disposal_rate_.values == np.array([0.244, 0.572, 0.704, 0.814, 0.910, 0.952, 0.982, 0.994, 1.000]))
#Exhibit VI Sheet 2
lhs = (e6_dr_tri.full_triangle_.cum_to_incr()-e6_tri['Closed Claim Counts'].cum_to_incr()).values.flatten()
rhs = np.array([
                                                        9.,  
                                                21.,    10.,  
                                        39.,    16.,    8.,  
                                109.,   78.,    31.,    16.,  
                        235.,   103.,   74.,    29.,    15.,  
                177.,   155.,   68.,    48.,    19.,    10.,  
        160.,   133.,   116.,   51.,    36.,    15.,    7.,  
389.,   156.,   130.,   114.,   50.,    36.,    14.,    7.
])
assert np.all(abs(lhs[~np.isnan(lhs)] - rhs) < 1)
#Exhibit VI Sheet 4
assert np.all(e6_sevs_sel.values[...,:6].round(0) == np.array([11807.,15165.,26043.,35183.,41908.,62206.]))
#Exhibit VI Sheet 5
assert np.all(e6_s5.iloc[4].values.round(0) == np.array([62206.,70432.]))
#Exhibit VI Sheet 6
lhs = e6_ipsev_full_detrended[e6_ipsev_full_detrended.valuation > e6_ipsev.valuation_date].values.flatten()
rhs = np.array([
                                                        74709., 
                                                78444., 78444., 
                                        82367., 82367., 82367., 
                                76384., 86485., 86485., 86485., 
                        54032., 80203., 90809., 90809., 90809., 
                42549., 50682., 75230., 85179., 85179., 85179., 
        24803., 33508., 39912., 59244., 67079., 67079., 67079., 
15165., 26043., 35183., 41908., 62206., 70432., 70432., 70432., 
])
assert np.allclose(lhs[~np.isnan(lhs)], rhs, atol=1)
#Exhibit VI Sheet 7
assert np.allclose(
    e6_s7_ip.incr_to_cum().iloc[:,:,:,-1].values.round(0).flatten(),
    np.array([39192.,46869.,44479.,71906.,71684.,49913.,31805.,29828.]),
    rtol = .001
)
